# Building a Unified Summarization Dataset

In this section, we build a unified dataset by combining multiple publicly available summarization datasets.  
The goal is to prepare a balanced corpus that spans different domains such as scientific publications, news, government reports, and legal documents.  

The following datasets are considered:
- **PubMed** (`ccdv/pubmed-summarization`) – biomedical research articles and their abstracts.  
- **GovReport** (`ccdv/govreport-summarization`) – long government reports with human-written summaries.  
- **BillSum** (`FiscalNote/billsum`) – US congressional and state bills paired with reference summaries.  
- **CNN/XSum** – news articles with abstractive summaries. Data can be found at https://github.com/Tiiiger/benchmark_llm_summarization/blob/main/writer_summaries.json  
- **Newsroom** (`lil-lab/newsroom`) – news articles with extractive/abstractive summaries. The original dataset has been moved to https://lil.nlp.cornell.edu/newsroom/download/index.html  
- **BigPatent** (`NortheasternUniversity/big_patent`) – patent descriptions and their abstracts.  


In [ ]:
import json
import pandas as pd
from tqdm import tqdm
import nltk
import random
from sklearn.utils import shuffle
from datasets import load_dataset
from collections import Counter

# PubMed 
pubmed_ds = load_dataset("ccdv/pubmed-summarization", split="test")
pubmed_features = ["article", "abstract"]

# GovReport 
govreport_ds = load_dataset("ccdv/govreport-summarization", split="test")
govreport_features = ["report", "summary"]

# BillSum 
billsum_ds = load_dataset("FiscalNote/billsum", split="test")
billsum_features = ["text", "summary"]

# CNN/XSum 
with open("Dataset/CNN_XSum.json", "r", encoding="utf-8") as f:
    cnn_xsum_ds = json.load(f)
cnn_xsum_features = ["article", "summary"]

# Newsroom
with open("Dataset/newsroom_test.jsonl", "r", encoding="utf-8") as f:
    newsroom_ds = [json.loads(line) for line in f]
newsroom_features = ["text", "summary"]

# BigPatent 
# The BigPatent dataset is very large and may take a long time to download and process.
big_patent_ds = load_dataset(
    "NortheasternUniversity/big_patent", 
    split="test", 
    trust_remote_code=True
)
big_patent_features = ["description", "abstract"]


# Partitioning the Dataset

We partition each dataset into bins based on:
- **Summary word count** (e.g., ~50, 100, 150 words, etc.)
- **Summary sentence count** (1 to 6 sentences)
- **Compression ratio** (summary length compared to article length, e.g., 1/10, 1/5, 1/2)

This categorization allows us to retrieve balanced subsets of samples for benchmarking.  
Each selected sample is tagged with its bin membership.

In [3]:
def partition_dataset(df, dataset_name, features):
    word_bins = list(range(50, 301, 50))
    sentence_bins = list(range(1, 7))
    ratio_bins = [1/20, 1/10, 1/5, 1/4, 1/3, 1/2]

    word_partition = {bin_size: [] for bin_size in word_bins}
    sentence_partition = {bin_size: [] for bin_size in sentence_bins}
    ratio_partition = {ratio: [] for ratio in ratio_bins}

    ratio_names = {
        1/20: "1/20", 
        1/10: "1/10", 
        1/5: "1/5", 
        1/4: "1/4", 
        1/3: "1/3", 
        1/2: "1/2"
    }

    benchmark_data = []

    for i, item in tqdm(enumerate(df), total=len(df)):
        sample_name = f"{dataset_name}_{i}"
        article = item[features[0]]
        summary = item[features[1]]

        # Tokenization
        tokenized_summary = nltk.word_tokenize(summary)
        summary_word_count = len(tokenized_summary)

        tokenized_article = nltk.word_tokenize(article)
        article_word_count = len(tokenized_article)

        sentence_count = len(nltk.sent_tokenize(summary))
        compression_ratio = summary_word_count / article_word_count if article_word_count > 0 else None

        # Store sample metadata
        sample_data = {
            "id": sample_name,
            "document": article,
            "summary": summary,
            "document_word_count": article_word_count,
            "summary_word_count": summary_word_count,
            "summary_sentence_count": sentence_count,
            "compression_ratio": compression_ratio,
            "split_word": None,
            "split_sentence": None,
            "split_ratio": None  
        }

        # Word bin assignment
        for bin_size in word_bins:
            if bin_size - 5 <= summary_word_count <= bin_size + 5:
                word_partition[bin_size].append(sample_name)
                sample_data["split_word"] = f"{bin_size}"
                break

        # Sentence bin assignment
        if 1 <= sentence_count <= 6:
            sentence_partition[sentence_count].append(sample_name)
            sample_data["split_sentence"] = f"{sentence_count}"

        # Ratio bin assignment
        for r in ratio_bins:
            target_words = article_word_count * r  
            if abs(summary_word_count - target_words) <= 5:  
                ratio_partition[r].append(sample_name)
                sample_data["split_ratio"] = ratio_names[r]
                break    

        # Keep only samples that fall into at least one partition
        if sample_data["split_word"] or sample_data["split_sentence"] or sample_data["split_ratio"]:
            benchmark_data.append(sample_data)
        
    benchmark_data_df = pd.DataFrame(benchmark_data)
    return benchmark_data_df


In [4]:
# Partition datasets
pubmed_df = partition_dataset(pubmed_ds, "pubmed", pubmed_features)
print("PubMed:", pubmed_df.shape)

govreport_df = partition_dataset(govreport_ds, "govreport", govreport_features)
print("GovReport:", govreport_df.shape)

cnn_df = partition_dataset(cnn_xsum_ds, "cnn_xsum", cnn_xsum_features)
print("CNN/XSum:", cnn_df.shape)

billsum_df = partition_dataset(billsum_ds, "billsum", billsum_features)
print("BillSum:", billsum_df.shape)

newsroom_df = partition_dataset(newsroom_ds, "newsroom", newsroom_features)
print("Newsroom:", newsroom_df.shape)

big_patent_df = partition_dataset(big_patent_ds, "big_patent", big_patent_features)
print("BigPatent:", big_patent_df.shape)


100%|██████████| 6658/6658 [00:53<00:00, 124.54it/s]


PubMed: (3578, 10)


100%|██████████| 973/973 [00:22<00:00, 43.26it/s]


GovReport: (32, 10)


100%|██████████| 302/302 [00:00<00:00, 450.04it/s]


...

BillSum: (2679, 10)


100%|██████████| 108862/108862 [03:42<00:00, 490.00it/s] 


Newsroom: (108158, 10)


100%|██████████| 67072/67072 [15:32<00:00, 71.89it/s] 


BigPatent: (63020, 10)


In [5]:
# combining all datasets and adding a column to identify the source dataset
df_concat = pd.concat([pubmed_df, govreport_df, cnn_df, billsum_df,newsroom_df,big_patent_df], ignore_index=True)
df_concat['dataset'] = df_concat['id'].apply(lambda x: x.split("_")[0])

In [ ]:
# Group by dataset and compute stats
stats = (
    df_concat
    .groupby("dataset")["document_word_count"]
    .agg(["mean", pd.Series.mode])
    .reset_index()
)

# Rename columns for clarity
stats = stats.rename(columns={"mean": "avg_doc_len", "mode": "mode_doc_len"})

print(stats)


     dataset  avg_doc_len      mode_doc_len
0        big  5267.131672              3353
1    billsum  1572.720418               960
2        cnn   795.274834  [489, 967, 1091]
3  govreport  6631.906250              7199
4   newsroom   762.207742               136
5     pubmed  2909.149525      [1161, 1296]


### Dataset Selection Decision

After analyzing document length statistics, we observed that some datasets (particularly **BigPatent**, **GovReport**, and **PubMed**) contain extremely long documents.  
While these datasets are valuable for long-document summarization research, their length makes them impractical for GRPO (Grouped Reinforcement Preference Optimization) training, which involves multiple rollouts and preference comparisons per sample.  

To keep training stable and computationally feasible, we decided to exclude these high-length datasets from the unified corpus.  

For the same reason, we also skipped the ratio-based split computation during the final training phase.  
Since ratio calculations depend on document and summary lengths, including very long samples would have introduced extreme variance and significantly slowed down GRPO rollouts without improving the training signal.


In [8]:
# Datasets to remove due to excessive document length (unsuitable for GRPO)
datasets_to_remove = {"big", "govreport", "pubmed"}

# Keep only rows where dataset is NOT in the removal list
df_filtered = df_concat[~df_concat["dataset"].isin(datasets_to_remove)].reset_index(drop=True)

# Drop columns related to ratio-based analysis (not used in GRPO training)
cols_to_drop = ["compression_ratio", "split_ratio"]
df_filtered = df_filtered.drop(columns=[col for col in cols_to_drop if col in df_filtered.columns])

# Display filtering results
print("Before:", df_concat["dataset"].unique())
print("After:", df_filtered["dataset"].unique())
print("Shape before:", df_concat.shape, " -> after:", df_filtered.shape)
print("\nDropped columns:", [col for col in cols_to_drop if col not in df_filtered.columns])


Before: ['pubmed' 'govreport' 'cnn' 'billsum' 'newsroom' 'big']
After: ['cnn' 'billsum' 'newsroom']
Shape before: (177769, 11)  -> after: (111139, 9)

Dropped columns: ['compression_ratio', 'split_ratio']


In [9]:
def print_distribution(df, column, label):
    counts = df[column].value_counts()
    print(f"\n{label} Distribution:")
    print(counts)
    print(f"Total samples: {counts.sum():,}")

print_distribution(df_filtered, 'split_word', 'Word-level Split')
print_distribution(df_filtered, 'split_sentence', 'Sentence-level')



Word-level Split Distribution:
split_word
50     5060
100     429
150     353
200     159
250     107
300      92
Name: count, dtype: int64
Total samples: 6,200

Sentence-level Distribution:
split_sentence
1    82889
2    18512
3     4912
4     2226
5     1395
6      818
Name: count, dtype: int64
Total samples: 110,752


From these results, we can see that:
- Some word-length categories (especially above 150 words) have relatively few samples,  
  indicating that the dataset is somewhat imbalanced in terms of document size.  
- The majority of examples are shorter, single-sentence summaries, which aligns well with GRPO’s need for efficient rollouts.  
- However, the lack of larger samples limits diversity, so we maintain a modest target size for training and evaluation.


### Split Generation Algorithm

To construct reproducible and balanced training subsets, we use a custom algorithm that partitions the filtered dataset into train, validation, and test splits.  
Unlike a random global split, this procedure ensures that each subset fairly represents the underlying distributions of sentence and word-level characteristics across different domains.



In [ ]:
SEED = 42
random.seed(SEED)

train_size = 275
val_size = 25
test_size = 100

split_columns = ['split_word', 'split_sentence']

final_splits = {"train": [], "val": [], "test": []}

for col in split_columns:
    unique_values = df_filtered[col].dropna().unique()

    for val in unique_values:
        subset_df = df_filtered[df_filtered[col] == val]
        if subset_df.empty:
            continue

        print(f"\n=== Processing {col} = {val} ===")

        df = shuffle(subset_df, random_state=SEED)
        domains = df['dataset'].unique()

        train_df = pd.DataFrame(columns=df.columns)
        val_df = pd.DataFrame(columns=df.columns)
        test_df = pd.DataFrame(columns=df.columns)

        remaining_train, remaining_val, remaining_test = train_size, val_size, test_size
        available_domains = list(domains)

        while (remaining_train > 0 or remaining_val > 0 or remaining_test > 0) and available_domains:
            domains_to_remove = []
            for domain in available_domains:
                domain_df = df[df['dataset'] == domain]
                used_samples = (
                    len(train_df[train_df['dataset'] == domain]) +
                    len(val_df[val_df['dataset'] == domain]) +
                    len(test_df[test_df['dataset'] == domain])
                )
                if used_samples >= len(domain_df):
                    domains_to_remove.append(domain)
                    continue

                remaining_samples = domain_df.iloc[used_samples:].copy()

                if remaining_train > 0 and len(remaining_samples) > 0:
                    to_add = min(1, len(remaining_samples), remaining_train)
                    train_df = pd.concat([train_df, remaining_samples[:to_add]])
                    remaining_samples = remaining_samples[to_add:]
                    remaining_train -= to_add

                if remaining_val > 0 and len(remaining_samples) > 0:
                    to_add = min(1, len(remaining_samples), remaining_val)
                    val_df = pd.concat([val_df, remaining_samples[:to_add]])
                    remaining_samples = remaining_samples[to_add:]
                    remaining_val -= to_add

                if remaining_test > 0 and len(remaining_samples) > 0:
                    to_add = min(1, len(remaining_samples), remaining_test)
                    test_df = pd.concat([test_df, remaining_samples[:to_add]])
                    remaining_samples = remaining_samples[to_add:]
                    remaining_test -= to_add

            for domain in domains_to_remove:
                available_domains.remove(domain)

        # Final shuffle with fixed seed
        train_df = train_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        val_df   = val_df.sample(frac=1, random_state=SEED).reset_index(drop=True)
        test_df  = test_df.sample(frac=1, random_state=SEED).reset_index(drop=True)

        # Tag with partition info
        safe_val = str(val).replace(".", "_").replace("/", "_")
        train_df["split"] = f"{col}_{safe_val}"
        val_df["split"]   = f"{col}_{safe_val}"
        test_df["split"]  = f"{col}_{safe_val}"

        # Append to final dictionary
        final_splits["train"].extend(train_df.to_dict(orient="records"))
        final_splits["val"].extend(val_df.to_dict(orient="records"))
        final_splits["test"].extend(test_df.to_dict(orient="records"))

with open("Dataset/summary_corpus_merged.json", "w", encoding="utf-8") as f:
    json.dump(final_splits, f, indent=2, ensure_ascii=False)

print("Final dataset with splits saved to summary_corpus_merged.json")



=== Processing split_word = 50 ===

=== Processing split_word = 100 ===

=== Processing split_word = 200 ===

=== Processing split_word = 300 ===

=== Processing split_word = 150 ===

=== Processing split_word = 250 ===

=== Processing split_sentence = 3 ===

=== Processing split_sentence = 2 ===

=== Processing split_sentence = 4 ===

=== Processing split_sentence = 1 ===

=== Processing split_sentence = 5 ===

=== Processing split_sentence = 6 ===
Final dataset with splits saved to summary_corpus_merged.json


### Final Split Verification

As a final step, we verify the integrity of the generated splits (`train`, `val`, and `test`) from the saved JSON file.  

The output below summarizes how many samples fall into each split category within every partition of the final corpus.


In [17]:
with open("Dataset/summary_corpus_merged.json", "r", encoding="utf-8") as f:
    data = json.load(f)

# Count how many of each split value per dataset split
for set_name in ["train", "val", "test"]:
    split_values = [sample["split"] for sample in data[set_name] if "split" in sample]
    counts = Counter(split_values)
    
    print(f"\n=== {set_name.upper()} ===")
    for split_value, count in counts.items():
        print(f"{split_value}: {count}")



=== TRAIN ===
split_word_50: 275
split_word_100: 275
split_word_200: 67
split_word_300: 34
split_word_150: 228
split_word_250: 42
split_sentence_3: 275
split_sentence_2: 275
split_sentence_4: 275
split_sentence_1: 275
split_sentence_5: 275
split_sentence_6: 275

=== VAL ===
split_word_50: 25
split_word_100: 25
split_word_200: 25
split_word_300: 25
split_word_150: 25
split_word_250: 25
split_sentence_3: 25
split_sentence_2: 25
split_sentence_4: 25
split_sentence_1: 25
split_sentence_5: 25
split_sentence_6: 25

=== TEST ===
split_word_50: 100
split_word_100: 100
split_word_200: 67
split_word_300: 33
split_word_150: 100
split_word_250: 40
split_sentence_3: 100
split_sentence_2: 100
split_sentence_4: 100
split_sentence_1: 100
split_sentence_5: 100
split_sentence_6: 100
